In [2]:
!pip install ollama

In [3]:
import json
import uuid
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import ollama

/Users/sterinsaji/miniconda3/envs/rag_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
PDF_PATHS = [
    "docs/NIST.CSWP.29.pdf",
    "docs/NIST.SP.800-53r5.pdf",
    "docs/NIST.SP.800-171r3.pdf",
    "docs/OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf",
    "docs/rfc9110.pdf",
    "docs/wellarchitected-framework.pdf"
]

In [ ]:
import datetime
import json

EXPERIMENT_CONFIG = {
    "timestamp": str(datetime.datetime.now()),
    "model": "qwen2.5:7b",
    "temperature": 0.0,
    "chunk_size": 800,
    "chunk_overlap": 120,
    "prompt_version": "v1",
    "pdfs": PDF_PATHS
}

with open("experiment_config_ollama.json", "w") as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=2)

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

all_chunks = []

for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = f"{pdf_path}_chunk_{i}"
        chunk.metadata["source_doc"] = pdf_path

    all_chunks.extend(chunks)

print("Total chunks:", len(all_chunks))

Total chunks: 7513


In [9]:
chunk_dump = [
    {
        "chunk_id": c.metadata["chunk_id"],
        "source_doc": c.metadata["source_doc"],
        "text": c.page_content
    }
    for c in all_chunks
]

with open("chunks_ollama.jsonl", "w", encoding="utf-8") as f:
    for c in chunk_dump:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

In [10]:
QA_PROMPT = """
You are generating factual QA pairs for evaluating hallucination detection in Retrieval-Augmented Generation systems.

RULES:
- Questions MUST be answerable ONLY from the provided text
- Do NOT use outside knowledge
- Avoid subjective or opinion questions
- Generate concise factual answers
- Include exact evidence spans
- Prefer:
  - definitions
  - requirements
  - procedures
  - identifiers
  - technical specifications
- Questions should be useful for evaluating:
  - RAGAS
  - SelfCheckGPT
  - Retrieval grounding
  - Hallucination detection

Generate EXACTLY 3 QA pairs.

Return ONLY valid JSON.

FORMAT:

[
  {{
    "question": "...",
    "answer": "...",
    "evidence": "...",
    "question_type": "definition | requirement | factual | procedure | identifier",
    "difficulty": "easy | medium | hard"
  }}
]

TEXT:
{chunk}
"""

In [14]:
def llm(prompt):
    response = ollama.chat(
        model="qwen2.5:7b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response["message"]["content"]

In [ ]:
def generate_qa_pairs(chunk):
    if not chunk.page_content.strip():
        return []

    prompt = QA_PROMPT.format(chunk=chunk.page_content[:3000])

    try:
        content = llm(prompt)

        # cleanup (important for JSON stability)
        content = content.replace("```json", "").replace("```", "").strip()

        qa_pairs = json.loads(content)

    except Exception as e:
        print("Failed chunk:", e)
        return []

    results = []

    for qa in qa_pairs:
        required = ["question", "answer", "evidence", "question_type", "difficulty"]

        if not all(k in qa for k in required):
            continue

        results.append({
            "id": str(uuid.uuid4()),
            "question": qa["question"],
            "answer": qa["answer"],
            "evidence": qa["evidence"],
            "question_type": qa["question_type"],
            "difficulty": qa["difficulty"],
            "chunk_id": chunk.metadata.get("chunk_id"),
            "source_doc": chunk.metadata.get("source_doc")
        })

    return results

In [20]:

import json
import os
from tqdm import tqdm

output_file = "qa_dataset_final.jsonl"

# =========================
# STEP 0: LOAD EXISTING PROGRESS
# =========================

done_chunks = set()

if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                done_chunks.add(obj["chunk_id"])
            except:
                pass

print(f"Already completed chunks: {len(done_chunks)}")


for i, chunk in enumerate(tqdm(all_chunks)):

    chunk_id = chunk.metadata.get("chunk_id")

    # =========================
    # SKIP IF ALREADY DONE
    # =========================
    if chunk_id in done_chunks:
        continue

    try:
        qa_pairs = generate_qa_pairs(chunk)

        # =========================
        # WRITE IMMEDIATELY (NO MEMORY BUFFER)
        # =========================
        with open(output_file, "a", encoding="utf-8") as f:
            for qa in qa_pairs:
                qa["chunk_id"] = chunk_id
                qa["source_doc"] = chunk.metadata.get("source_doc")
                f.write(json.dumps(qa) + "\n")

    except Exception as e:
        print(f"Error in chunk {chunk_id}: {e}")

    # =========================
    # OPTIONAL: progress log
    # =========================
    if i % 50 == 0:
        print(f"Processed {i}/{len(all_chunks)} chunks")

print("DONE ✅")

Already completed chunks: 1099


 15%|█▍        | 1101/7513 [00:57<06:46, 15.79it/s]

Processed 1100/7513 chunks


 15%|█▌        | 1151/7513 [15:17<28:24:59, 16.08s/it]

Processed 1150/7513 chunks


...

Processed 7450/7513 chunks


100%|█████████▉| 7501/7513 [39:16:05<03:11, 15.92s/it]

Processed 7500/7513 chunks


100%|██████████| 7513/7513 [39:18:50<00:00, 18.84s/it]

DONE ✅
